# ReefNet audit

Run after `python -m reefwatch.data.ingest_reefnet`. Compare the totals with the ReefNet
docs (334,162 images, 924,626 points, 76 CoralNet sources + Al-Wajh). Write down differences in
`docs/DECISIONS.md`; don't "fix" them.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from omegaconf import OmegaConf

from reefwatch.data.ingest_reefnet import audit

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = OmegaConf.load(ROOT / "configs/data.yaml")
interim = ROOT / cfg.paths.interim
points = pd.read_parquet(interim / "reefnet_points.parquet")
summary = audit(points)
print({k: v for k, v in summary.items() if not k.startswith("counts_")})

## Rejected rows
Anything here was not a whole-number pixel coordinate. Look at a few before moving on.

In [ ]:
rej_path = interim / "ingest_rejects.csv"
rej = pd.read_csv(rej_path, dtype=str) if rej_path.exists() else pd.DataFrame()
print(len(rej), "rejected rows")
rej.head(10)

## Label counts
`Broad_category` decides whether ReefNet can train the L1 groups beyond hard coral.

In [ ]:
for c in ["Broad_category", "Train_or_Test", "Sorted_ReefNet_label"]:
    if f"counts_{c}" in summary:
        print(f"\n== {c}")
        print(summary[f"counts_{c}"].head(50).to_string())

## Points per image

In [ ]:
per_image = points.groupby(["CoralNet_source", "Image_name"]).size()
print(per_image.describe())
per_image.clip(upper=60).plot.hist(bins=60, title="Points per image (clipped at 60)")
plt.show()

## Row / Column check
Download one image from its CoralNet source by hand, set `IMAGE` below and look at the plot.
If the dots sit on the labelled organisms, `Row` is y and `Column` is x. If they look mirrored or
rotated, swap them and record it in `docs/DECISIONS.md`. Repeat for 3 images from different sources.

In [ ]:
IMAGE = ROOT / "data/raw/check/example.jpg"   # change me
SOURCE, NAME = "<CoralNet_source>", "<Image_name>"   # change me

if IMAGE.exists():
    img = plt.imread(IMAGE)
    pts = points[(points.CoralNet_source == SOURCE) & (points.Image_name == NAME)].head(5)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.scatter(pts["Column"], pts["Row"], s=80, facecolors="none", edgecolors="yellow")
    for _, p in pts.iterrows():
        plt.annotate(p.Sorted_ReefNet_label, (p.Column, p.Row), color="yellow")
    plt.title(f"{SOURCE}/{NAME}: Column as x, Row as y")
    plt.show()
else:
    print("Set IMAGE to a downloaded CoralNet image first.")